In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Config affichage
pd.set_option('display.max_columns', None)

# 1. Chargement des datasets
parts = pd.read_csv('../data/parts_master.csv')
po = pd.read_csv('../data/purchase_orders.csv')
quality = pd.read_csv('../data/quality_incidents.csv')
scenarios = pd.read_csv('../data/scenario_history.csv')
supply = pd.read_csv('../data/supply_chain_history.csv')

# 2. Aperçu rapide
print("=== TAILLES DES DATASETS ===")
print(f"Parts Master: {parts.shape}")
print(f"Purchase Orders: {po.shape}")
print(f"Quality Incidents: {quality.shape}")
print(f"Scenario History: {scenarios.shape}")
print(f"Supply Chain History: {supply.shape}")

print("\n=== COLONNES DE PURCHASE ORDERS ===")
print(po.columns.tolist())

print("\n=== COLONNES DE SUPPLY CHAIN HISTORY ===")
print(supply.columns.tolist())

=== TAILLES DES DATASETS ===
Parts Master: (300, 9)
Purchase Orders: (29666, 9)
Quality Incidents: (368, 8)
Scenario History: (6, 9)
Supply Chain History: (280800, 11)

=== COLONNES DE PURCHASE ORDERS ===
['po_id', 'supplier_id', 'site_id', 'part_id', 'order_date', 'promised_date', 'receipt_date', 'ordered_qty', 'received_qty']

=== COLONNES DE SUPPLY CHAIN HISTORY ===
['date', 'site_id', 'part_id', 'planned_maintenance', 'consumption_qty', 'on_hand_qty', 'backorder_qty', 'blocked_qty', 'forecast_qty', 'forecast_type', 'forecast_uplift_pct']


In [2]:
# 1. Fusion Purchase Orders avec Parts Master
df_ml = po.merge(parts, on='part_id', how='left')

# 2. Conversion des dates en datetime
df_ml['order_date'] = pd.to_datetime(df_ml['order_date'])
df_ml['promised_date'] = pd.to_datetime(df_ml['promised_date'])
df_ml['receipt_date'] = pd.to_datetime(df_ml['receipt_date'])

# 3. Création des variables de délai (Features)
df_ml['promised_lead_time'] = (df_ml['promised_date'] - df_ml['order_date']).dt.days
df_ml['actual_lead_time'] = (df_ml['receipt_date'] - df_ml['order_date']).dt.days

# 4. Création de la variable CIBLE (TARGET)
# Is_Late = 1 si la date de réception dépasse la date promise, 0 sinon
df_ml['is_late'] = (df_ml['receipt_date'] > df_ml['promised_date']).astype(int)

# 5. Gestion des ratios d'exécution de commande
df_ml['qty_fill_rate'] = df_ml['received_qty'] / (df_ml['ordered_qty'] + 1e-5)

# Aperçu du dataset prêt pour le ML
print(f"Dataset prêt : {df_ml.shape}")
print(f"Taux de retards de livraison : {df_ml['is_late'].mean() * 100:.2f}%")
df_ml[['po_id', 'part_id', 'promised_lead_time', 'actual_lead_time', 'is_late']].head()

Dataset prêt : (29666, 21)
Taux de retards de livraison : 55.85%


,po_id,part_id,promised_lead_time,actual_lead_time,is_late
0,PO000001,P00001,26,27,1
1,PO000002,P00001,27,29,1
2,PO000003,P00001,26,30,1
3,PO000004,P00001,29,31,1
4,PO000005,P00001,25,26,1


In [3]:
# Nettoyage minimalist
features_to_keep = [
    'po_id', 'supplier_id', 'site_id', 'part_id', 
    'ordered_qty', 'promised_lead_time', 'qty_fill_rate', 'is_late'
]

# Si des colonnes de parts_master sont intéressantes (ex: unit_cost, category), ajoute-les ici
if 'category' in parts.columns:
    features_to_keep.append('category')

df_clean = df_ml[features_to_keep].dropna()

# Sauvegarde pour le Notebook 02
df_clean.to_csv('../data/processed_mro_ml.csv', index=False)
print("Fichier '../data/processed_mro_ml.csv' enregistré avec succès !")

Fichier '../data/processed_mro_ml.csv' enregistré avec succès !
